In [1]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import random_split
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_dataset
from core.data.dataset import create_dataloaders
from core.data.transforms import create_normalizer_from_data
from core.model.bert import BertForMaskedModeling
from core.training.sampler import create_kde_sampler
from core.training.pretrainer import setup_training
from core.logger import print_data_summary, log_model_summary

In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BERT Configuration")

PyTorch version: 2.8.0+cu126
Using device: cuda
Working directory: /home/jessiez/osu_corpora

--- Loaded BERT Configuration ---
data:
  db_path: ./data/beatmap_dataset_test/
  max_seq_len: 1023
  val_split: 0.1
  max_samples_per_class:
    aim: 1500
    tech: 1500
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
  local_attention_window: 256
components:
  use_flash_attention: true
  compile_model: true
  compile_mode: default
pretraining:
  batch_size: 8
  num_epochs: 5
  learning_rate: 0.0005
  min_lr: 1.0e-06
  cooldown_type: cosine
  weight_decay: 0.05
  warmup_ratio: 0.05
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./checkpoints
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 8
  masking_ratio: 0.25
  mean_span_length: 3
  sampling:
    method: kde
    kde_bandwidth: 0.5
    num_bins: 200
    expand_for_augmentation: false
finetuning:
  batch_size: 8
  num_epochs: 3
  learning_rate: 3.0e-05
  min_lr: 1.0e-06
  cooldown_type: l

In [3]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['data']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_ratings, loaded_ids = load_dataset(
    DATASET_PATH, 
    max_seq_len=config['data']['max_seq_len']
)

print_data_summary(all_beatmaps_data)

Using database: ./data/beatmap_dataset_test/
Loading raw data from Parquet dataset...
Loading and processing curve point data with filters...
Loaded 9812 beatmaps and 8059989 hit objects.
Engineering features for all beatmaps (vectorized)...
Converting processed dataframes to tensors...


100%|██████████| 9808/9808 [00:00<00:00, 262622.63it/s]


Recalculating difficulty ratings for sequences truncated to 1023...


Recalculating Stars: 100%|██████████| 1/1 [00:00<00:00, 7157.52it/s]


Applying log transforms...


...

Running final data integrity check...


Validating Tensors:  57%|█████▋    | 5542/9807 [00:00<00:00, 13226.29it/s]

Inf found in vectors: True, metadata: False


Validating Tensors: 100%|██████████| 9807/9807 [00:00<00:00, 12979.92it/s]

Finished loading and processing all data.

--- Data Summary ---
Total beatmaps: 9806
Vector dimension: 20
Metadata dimension: 6
Sequence length - Min: 48, Max: 1023, Avg: 709.6
--------------------


In [4]:
val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

print(f"Data split: {len(train_data)} training, {len(val_data)} validation")

train_data_list = [train_data.dataset[i] for i in train_data.indices]
val_data_list = [val_data.dataset[i] for i in val_data.indices]

train_difficulty_ratings = difficulty_ratings[train_data.indices]

sampler = create_kde_sampler(
    train_difficulty_ratings,
    bandwidth=config['pretraining']['sampling']['kde_bandwidth'],
    expand_for_augmentation=config['pretraining']['sampling']['expand_for_augmentation'],
    num_bins=config['pretraining']['sampling'].get('num_bins', 100),
)

normalizer = create_normalizer_from_data(
    train_data_list,
    # include_augmentation=config['pretraining']['sampling']['expand_for_augmentation']
)

vector_stats = normalizer.get_vector_stats()
meta_stats = normalizer.get_metadata_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")
print(f"Metadata normalization stats for {len(meta_stats)} fields")

Data split: 8826 training, 980 validation
Creating optimized KDE sampler with bandwidth=0.5, bins=200...
KDE sampling - Min weight: 0.3670, Max weight: 1502.0287
Calculating normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
--------------------------------------------------------------------------------
Field Name             Type         Param 1      Param 2      Description
--------------------------------------------------------------------------------
distance_diff          log+norm     4.2938       1.5105       Distance from previous hit object's end point (jump distance)
velocity               log+norm     0.6292       0.3652       Velocity to previous hit object (pixels/ms)
cos_flow_angle         mean/std     0.1488       0.7585       Cosine of angle between previous object's exit path and current object's arrival path
sin_flow_angle         mean/std     -0.0008      0.6345       Sine of angle between previous object's exit path an

In [5]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['pretraining']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}, meta={sample_batch[2].shape}")

Created dataloaders with batch size: 8
Sample batch shapes: vectors=torch.Size([8, 1023, 20]), mask=torch.Size([8, 1023]), meta=torch.Size([8, 6])


In [6]:
model = BertForMaskedModeling.from_config(config, device)
log_model_summary(model)

print("\nRunning a test forward pass with mixed precision (autocast)...")
with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        sample_vectors, sample_mask, sample_metadata = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)
        sample_metadata = sample_metadata.to(device)

        predictions, targets, _ = model(sample_vectors, sample_metadata, sample_mask)

print("\nBERT model created and tested successfully!")

Compiling BERT model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 25.44M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6
Flash Attention: True
------------------------------

--- MLM Head Information ---
Task: Masked Modeling
Masking Ratio: 0.25
Model Compiled: True
------------------------------

Running a test forward pass with mixed precision (autocast)...


W0927 19:54:28.950000 80790 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1436] [7/0_1] Not enough SMs to use max_autotune_gemm mode



BERT model created and tested successfully!


In [7]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device, normalizer
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1 
        print(f"Loaded checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting pretraining from scratch")

print(f"Pretraining setup complete. Starting from epoch {start_epoch + 1}")
print(f"Total epochs: {config['pretraining']['num_epochs']}")

Scheduler: WSD with 34 warmup, 69 stable, 587 decay steps.
Cooldown type: cosine, Min LR Ratio: 0.0020
Trainer initialized - AMP: True, Device: cuda, Grad Accum: 8
Effective batch size: 64
Pretraining setup complete. Starting from epoch 1
Total epochs: 5


In [8]:
print("\nStarting BERT pretraining...")
print(f"BERT Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")

expand = config['pretraining']['sampling'].get('expand_for_augmentation', True)

if expand:
    effective_train_size = len(train_data) * 4
    print(f"Pretraining samples: {len(train_data)} base maps -> {effective_train_size}")
else:
    effective_train_size = len(train_data)
    print(f"Pretraining samples: {len(train_data)} base maps")

metrics_tracker = trainer.train(start_epoch)

print("\nBERT training completed!")


Starting BERT pretraining...
BERT Model: 6 layers, 512 dimensions
Pretraining samples: 8826 base maps

--- Starting Training ---
Epochs: 1 to 5
Batch Size: 8
Learning Rate: 0.0005
------------------------------------------------------------


Epoch 1 [Train]:   0%|          | 0/138 [00:00<?, ?it/s]

Epoch 1 [Validate]:   0%|          | 0/123 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 13.1807 | Val Loss: 12.3827 | LR: 4.96e-04 | Time: 137.25s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE / Error     | Mean (True)  | Std (True)  
  -----------------------+-----------------+--------------+-------------
  distance_diff          | 0.5888          | 0.0076       | 0.9921      
  velocity               | 0.5802          | 0.0001       | 0.9877      
  slider_num_anchors     | 0.7013          | 0.0008       | 0.9972      
  slider_pixel_length    | 0.6783          | 0.0001       | 0.9999      
  slide_length           | 0.6943          | 0.0002       | 1.0000      
  slider_repeats         | 0.3181          | -0.0006      | 0.9944      
  slider_velocity        | 0.6917          | -0.0050      | 0.9873      
  slider_tortuosity      | 0.3145          | -0.0027      | 0.9774      
  hard_anchor_ratio   

Epoch 2 [Train]:   0%|          | 0/138 [00:00<?, ?it/s]

...

Epoch 4 [Validate]:   0%|          | 0/123 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 7.9192 | Val Loss: 10.1396 | LR: 6.60e-05 | Time: 125.25s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE / Error     | Mean (True)  | Std (True)  
  -----------------------+-----------------+--------------+-------------
  distance_diff          | 0.4558          | 0.0098       | 0.9896      
  velocity               | 0.4741          | 0.0010       | 0.9846      
  slider_num_anchors     | 0.5175          | -0.0004      | 0.9942      
  slider_pixel_length    | 0.5299          | 0.0002       | 1.0000      
  slide_length           | 0.5348          | 0.0004       | 1.0001      
  slider_repeats         | 0.2210          | -0.0023      | 0.9804      
  slider_velocity        | 0.4892          | -0.0058      | 0.9863      
  slider_tortuosity      | 0.2490          | -0.0051      | 0.9243      
  hard_anchor_ratio    

Epoch 5 [Train]:   0%|          | 0/138 [00:00<?, ?it/s]

Epoch 5 [Validate]:   0%|          | 0/123 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 7.6189 | Val Loss: 10.1155 | LR: 1.00e-06 | Time: 119.47s
                      DETAILED VALIDATION REPORT                       
----------------------------------------------------------------------
 CONTINUOUS FEATURES:
  Feature                | MAE / Error     | Mean (True)  | Std (True)  
  -----------------------+-----------------+--------------+-------------
  distance_diff          | 0.4382          | 0.0101       | 0.9913      
  velocity               | 0.4668          | 0.0019       | 0.9851      
  slider_num_anchors     | 0.5153          | 0.0033       | 0.9985      
  slider_pixel_length    | 0.5262          | 0.0029       | 1.0017      
  slide_length           | 0.5278          | 0.0026       | 1.0013      
  slider_repeats         | 0.2289          | 0.0009       | 0.9876      
  slider_velocity        | 0.4815          | -0.0032      | 0.9853      
  slider_tortuosity      | 0.2451          | -0.0003      | 0.9684      
  hard_anchor_ratio    